# Experiment 3 · Uncertainty-Aware Adaptive Computation (SageMaker)

**Research question (RQ2):** can we spend expensive computation (sliced inference, or a higher overlap) **only on the images that need it**, and keep almost all of the accuracy of always slicing at a much lower cost?

This notebook **does not train or run any model**. Every image already has saved predictions and measured per-image latency for a cheap pass (*light*, whole frame) and one or more expensive passes (*strong*, sliced; e.g. the best overlap setting from Experiment 1B). We simulate the decision offline:

1. Run the light pass on every image and compute an **image-level uncertainty signal** from it (the Experiment 2 signals).
2. If the signal is above a threshold, also run the strong pass and use its detections instead. The cost of an escalated image is **light + strong** latency, because the light pass already happened.

Policies compared on the **policy-test** images (the threshold is chosen on the **calibration** images only):

| Policy | Meaning |
|---|---|
| `light_only` | whole frame for every image |
| `strong_only_<k>` | always sliced (each strong setting) |
| `adaptive_uncertainty` | escalate when the uncertainty signal ≥ threshold |
| `random_same_fraction` | escalate the same number of images at random (1000 draws) |
| `size_prior_same_fraction` | escalate images with the most small predicted boxes (is it more than a size heuristic?) |
| `oracle_same_fraction` | escalate the images that truly gain most (upper bound, not deployable) |
| `graded_overlap` | with ≥ 2 strong settings: low uncertainty → whole, medium → cheaper slicing, high → most expensive slicing, at **no more cost** than the binary policy |

Threshold rule: on calibration images, choose the **smallest escalation fraction that keeps ≥ 90% of the F1 gain of always slicing** (`target_gain_retention`). Paired image-level bootstrap gives 95% CIs for the differences. Budget curves show F1 at every escalation fraction from 0% to 100%.

The calibration / policy-test split is the same deterministic 219 / 329 split as Experiment 2 (or reuse its `image_split.csv`). All outputs stay in SageMaker under `results/exp3_adaptive_computation/<run_id>/`; ClearML gets tables, scalars, figures and a small inventory in a **new** project.

In [1]:
# Restart the kernel after the first installation if prompted, then run all cells top-to-bottom.
%pip install -q "clearml>=1.16,<3" "pycocotools>=2.0.7,<3" "numpy>=1.24,<3" "pandas>=2,<3" "matplotlib>=3.7,<4" "tqdm>=4.66,<5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass

os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

os.environ["CLEARML_API_ACCESS_KEY"] = getpass("ClearML access key: ")
os.environ["CLEARML_API_SECRET_KEY"] = getpass("ClearML secret key: ")

print("ClearML credentials set for this kernel.")

ClearML access key:  ········
ClearML secret key:  ········


ClearML credentials set for this kernel.


In [3]:
import csv
import hashlib
import json
import os
import platform
import random
import shutil
import time
import uuid
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm

@dataclass
class Config:
    # The project root is the SAME current directory used by notebook 00 and Experiment 1.
    project_root: str = "SageMaker_VisDrone_OVD_2026"
    task_name_prefix: str = "Experiment 3 - Uncertainty-Aware Adaptive Computation"
    clearml_project: str = "SageMaker-Aerial-OVD-2026/Uncertainty-Adaptive-Computation-2026"
    run_label: str = "YOLOE_26 zero-shot"

    # Condition spec "<model_family>/<run_id>::<condition>" -> results/<model_family>/<run_id>/predictions_<condition>.json
    # light  = the cheap pass that is always run (whole frame).
    # strong = one or more expensive passes of the SAME model. Put the best setting from Experiment 1B FIRST:
    #          the binary policy uses strong[0]; with >= 2 entries the graded-overlap policy is also run.
    light: str = "yoloe_26/20261028T153524Z_5dd65a8e::zero_shot__whole"
    strong: tuple = ("yoloe_26/20261028T153524Z_5dd65a8e::zero_shot__sliced_overlap_20",)

    signal: str = "auto"                     # "auto" = best candidate on calibration images, or a signal name.
    candidate_signals: tuple = ("n_band", "band_ratio", "mean_u", "mean_entropy")
    target_gain_retention: float = 0.90      # Keep >= 90% of the F1 gain of always running strong[0].

    operating_threshold: float = 0.25        # Same operating threshold as Experiment 1.
    analysis_floor: float = 0.05
    band_low: float = 0.10
    evaluation_iou: float = 0.50
    calibration_fraction: float = 0.40       # 548 val images -> 219 calibration / 329 policy-test.
    split_seed: int = 42
    split_csv: str = ""                      # Optional: Experiment 2 image_split.csv.
    exclude_early_stopping_images: bool = False
    allow_smoke_sources: bool = False
    budget_step: float = 0.05
    random_draws: int = 1000
    random_curve_draws: int = 200
    random_ap_draws: int = 20
    bootstrap_samples: int = 1000
    max_detections_per_image: int = 500
    seed: int = 42

CFG = Config()
assert 0 < CFG.analysis_floor <= CFG.band_low < CFG.operating_threshold < 1
assert 0 < CFG.target_gain_retention <= 1 and 0 < CFG.budget_step <= 0.5
assert len(CFG.strong) >= 1

random.seed(CFG.seed)
np.random.seed(CFG.seed)
ROOT = (Path.cwd().resolve() / CFG.project_root)
DATA_ROOT = ROOT / "visdrone_det_original"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RESULTS_DIR = ROOT / "results" / "exp3_adaptive_computation" / RUN_ID
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=False)
FIGURES_DIR.mkdir()
print("Run:", RUN_ID)
print("Light:", CFG.light)
print("Strong:", CFG.strong)
print("Persistent local results:", RESULTS_DIR)
print("Free space (GiB):", round(shutil.disk_usage(ROOT).free / 1024**3, 2))
print("New ClearML project:", CFG.clearml_project)

Run: 20261029T130521Z_615f90ed
Light: yoloe_26/20261028T153524Z_5dd65a8e::zero_shot__whole
Strong: ('yoloe_26/20261028T153524Z_5dd65a8e::zero_shot__sliced_overlap_20',)
Persistent local results: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/exp3_adaptive_computation/20261029T130521Z_615f90ed
Free space (GiB): 6.54
New ClearML project: SageMaker-Aerial-OVD-2026/Uncertainty-Adaptive-Computation-2026


## 1 · Load the *published* protocol, not a new random split

Exactly the same checks as the training notebooks: the finalized ClearML Dataset ID in `clearml_protocol_dataset_id.txt` is the only source of category IDs, prompts and the 7-base / 3-novel split. The fingerprint must match before any analysis runs.

In [4]:
from clearml import Dataset, Task

id_file = ROOT / "clearml_protocol_dataset_id.txt"
if not id_file.is_file():
    raise FileNotFoundError(f"Run notebook 00 first; missing {id_file}")
PROTOCOL_ID = id_file.read_text(encoding="utf-8").strip()
ds = Dataset.get(dataset_id=PROTOCOL_ID)
if not ds.is_final():
    raise RuntimeError("Protocol Dataset is not finalized. Do not analyse mutable splits.")
PROTOCOL_DIR = Path(ds.get_local_copy()).resolve()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

required = {"categories_and_prompts.json", "protocol.json", "summary.json",
            "train.csv", "val.csv", "test.csv", "train_base_safe.csv", "protocol_fingerprint.txt"}
if not required.issubset({p.name for p in PROTOCOL_DIR.iterdir() if p.is_file()}):
    raise RuntimeError("ClearML protocol Dataset is missing a required v1 file.")
digest = hashlib.sha256()
for name in sorted(required - {"protocol_fingerprint.txt"}):
    content = (PROTOCOL_DIR / name).read_bytes()
    digest.update(name.encode() + b"\x00" + content + b"\x00")
FINGERPRINT = digest.hexdigest()
if FINGERPRINT != (PROTOCOL_DIR / "protocol_fingerprint.txt").read_text().strip():
    raise RuntimeError("Protocol contents do not match their published fingerprint.")
meta_fingerprint = (ds.get_metadata() or {}).get("protocol_fingerprint")
if meta_fingerprint != FINGERPRINT:
    raise RuntimeError("ClearML Dataset metadata fingerprint does not match its files.")
spec = json.loads((PROTOCOL_DIR / "protocol.json").read_text(encoding="utf-8"))
cats = json.loads((PROTOCOL_DIR / "categories_and_prompts.json").read_text(encoding="utf-8"))
if spec["version"] != "1.0.0":
    raise RuntimeError("This notebook expects the published protocol v1.0.0.")
CATEGORY_IDS = sorted(int(k) for k in cats["official_category_id_to_name"])
CATEGORY_NAMES = [cats["official_category_id_to_name"][str(k)] for k in CATEGORY_IDS]
PROMPTS = [cats["prompt_by_category_name"][name] for name in CATEGORY_NAMES]
BASE_NAMES = cats["base_category_names"]
NOVEL_NAMES = cats["novel_category_names"]
BASE_IDS = set(cats["base_official_ids"])
NOVEL_IDS = set(cats["novel_official_ids"])
assert len(CATEGORY_IDS) == 10 and set(CATEGORY_IDS) == BASE_IDS | NOVEL_IDS
assert BASE_IDS.isdisjoint(NOVEL_IDS) and len(set(PROMPTS)) == 10
GROUPS = {"all": set(CATEGORY_IDS), "base": BASE_IDS, "novel": NOVEL_IDS}

with (PROTOCOL_DIR / "val.csv").open(newline="", encoding="utf-8") as stream:
    val_rows = list(csv.DictReader(stream))
assert len(val_rows) == spec["expected_images"]["val"]
VAL_RELPATHS = {r["image_relpath"] for r in val_rows}
print("Frozen Dataset ID:", PROTOCOL_ID, "| fingerprint:", FINGERPRINT)
print("Base:", BASE_NAMES, "| Novel:", NOVEL_NAMES)

Frozen Dataset ID: 738dce9e1a5440dab34195418a997a20 | fingerprint: 158cb2c904016fa324c236b998393b1a94c176a4eabedb159c76bdeffc66c195
Base: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'motor'] | Novel: ['tricycle', 'awning-tricycle', 'bus']


## 3 · Create a NEW ClearML experiment project

Only tables, scalars, figures and a small inventory are sent to ClearML.

In [5]:
if Task.current_task() is not None:
    raise RuntimeError("An unrelated ClearML task is active. Restart the kernel before this notebook.")

task = Task.init(
    project_name=CFG.clearml_project,
    task_name=f"{CFG.task_name_prefix} | {CFG.run_label} | val | {RUN_ID}",
    reuse_last_task_id=False,
    auto_connect_frameworks=False,
)
logger = task.get_logger()
cfg_logged = asdict(CFG)
task.connect(cfg_logged, name="experiment_config")
task.connect({"protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
              "local_results_dir": str(RESULTS_DIR), "run_id": RUN_ID,
              "python": platform.python_version()}, name="provenance")
print("ClearML task:", task.id)

def record_local_file(path):
    path = Path(path).resolve()
    if not path.is_file() or not path.is_relative_to(RESULTS_DIR):
        raise ValueError(f"Expected a stored file under results/: {path}")
    return {"local_path": str(path), "file_uri": path.as_uri(),
            "bytes": path.stat().st_size, "sha256": sha256_file(path)}

def write_json(filename, value):
    destination = RESULTS_DIR / filename
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite an earlier output: {destination}")
    destination.write_text(json.dumps(clean_json(value), indent=2, ensure_ascii=False, allow_nan=False) + "\n",
                           encoding="utf-8")
    return destination

def write_csv(filename, frame):
    destination = RESULTS_DIR / filename
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite an earlier output: {destination}")
    frame.to_csv(destination, index=False)
    return destination

def log_table(title, frame):
    logger.report_table(title=title, series="val", iteration=0, table_plot=frame)

def log_figure(title, series, figure, filename):
    path = FIGURES_DIR / filename
    figure.savefig(path, dpi=150, bbox_inches="tight")
    logger.report_matplotlib_figure(title=title, series=series, figure=figure, iteration=0, report_image=True)
    plt.close(figure)
    return path

LOCAL_INDEX = {"results": {}}

ClearML Task: created new task id=dc3b63d99f024fb8a7516d277a0824f1
ClearML results page: https://app.clear.ml/projects/16e404d1591040f286673c00341ba856/tasks/dc3b63d99f024fb8a7516d277a0824f1/output/log
ClearML task: dc3b63d99f024fb8a7516d277a0824f1
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


## 2 · Load saved predictions from earlier runs (no retraining)

Each source run is a folder `results/<model_family>/<run_id>/` written by an Experiment 1 notebook. It must contain `config_and_protocol.json`, `coco_ground_truth.json`, `predictions_<condition>.json` and `latency_<condition>.json`. The loader refuses runs from a different protocol fingerprint, smoke runs (unless explicitly allowed) and runs evaluated on a different image list. Matching follows the Experiment 1 operating rule exactly: class-aware, highest score first, IoU ≥ 0.5.

In [6]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def clean_json(value):
    """Replace NaN/inf with None so json.dumps(allow_nan=False) never fails."""
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

def size_bin(area):
    if area < 16**2:
        return "tiny"
    if area < 32**2:
        return "small"
    if area < 96**2:
        return "medium"
    return "large"

def group_of(category_id):
    return "base" if category_id in BASE_IDS else "novel"

_SOURCE_CACHE = {}

def load_source_run(family, run_id):
    """Load config, ground truth and the image list of one saved Experiment 1 run."""
    key = (family, run_id)
    if key in _SOURCE_CACHE:
        return _SOURCE_CACHE[key]
    folder = ROOT / "results" / family / run_id
    if not folder.is_dir():
        raise FileNotFoundError(f"Source results folder not found: {folder}")
    src_cfg = read_json(folder / "config_and_protocol.json")
    if src_cfg.get("protocol_fingerprint") != FINGERPRINT:
        raise RuntimeError(f"{folder}: produced with a different protocol fingerprint.")
    run_mode = src_cfg.get("config", {}).get("run_mode")
    if run_mode != "full" and not CFG.allow_smoke_sources:
        raise RuntimeError(f"{folder}: run_mode={run_mode!r}. Smoke results are not paper results; "
                           "set allow_smoke_sources=True only for a pipeline check.")
    if src_cfg.get("config", {}).get("eval_split", "val") != "val":
        raise RuntimeError(f"{folder}: this analysis is a development analysis on val only.")
    coco_gt = read_json(folder / "coco_ground_truth.json")
    id_to_rel = {img["id"]: img["file_name"] for img in coco_gt["images"]}
    rel_to_id = {rel: i for i, rel in id_to_rel.items()}
    eval_rels = list(src_cfg["eval_image_relpaths"])
    if set(eval_rels) != set(id_to_rel.values()):
        raise RuntimeError(f"{folder}: ground-truth images differ from eval_image_relpaths.")
    if not set(eval_rels) <= VAL_RELPATHS:
        raise RuntimeError(f"{folder}: evaluation images are not all in the frozen val manifest.")
    gt = defaultdict(list)
    for ann in coco_gt["annotations"]:
        x, y, w, h = ann["bbox"]
        gt[id_to_rel[ann["image_id"]]].append({
            "gt_id": ann["id"], "category_id": int(ann["category_id"]),
            "xyxy": [x, y, x + w, y + h], "area": float(w * h)})
    source = {"folder": folder, "config": src_cfg, "coco_gt": coco_gt, "id_to_rel": id_to_rel,
              "rel_to_id": rel_to_id, "eval_rels": eval_rels, "gt": gt,
              "early_stopping_rels": set(src_cfg.get("early_stopping_image_relpaths", []))}
    _SOURCE_CACHE[key] = source
    return source

def load_condition(family, run_id, condition):
    """Per-image predictions (xyxy) and per-image latency for one saved condition."""
    source = load_source_run(family, run_id)
    pred_file = source["folder"] / f"predictions_{condition}.json"
    lat_file = source["folder"] / f"latency_{condition}.json"
    for path in (pred_file, lat_file):
        if not path.is_file():
            raise FileNotFoundError(f"Missing saved output: {path}")
    preds = defaultdict(list)
    for p in read_json(pred_file):
        x, y, w, h = p["bbox"]
        preds[source["id_to_rel"][p["image_id"]]].append({
            "category_id": int(p["category_id"]), "score": float(p["score"]),
            "xyxy": [x, y, x + w, y + h], "bbox": [x, y, w, h], "area": float(w * h)})
    latency = {r["image_relpath"]: r for r in read_json(lat_file)}
    missing = set(source["eval_rels"]) - set(latency)
    if missing:
        raise RuntimeError(f"{lat_file}: latency missing for {len(missing)} evaluation images.")
    return {"family": family, "run_id": run_id, "condition": condition, "preds": preds,
            "latency": latency, "source": source}

def iou_matrix(a, b):
    a = np.asarray(a, float).reshape(-1, 4)
    b = np.asarray(b, float).reshape(-1, 4)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / np.where(union > 0, union, 1.0), 0.0)

def match_image(preds, gts, floor):
    """Greedy class-aware matching, highest score first, IoU >= CFG.evaluation_iou.

    Because matching is in descending score order, the TP/FP label of every
    prediction above any higher threshold (e.g. 0.25) is identical to what
    Experiment 1 obtained when it matched only predictions above that threshold.
    Returns kept predictions (sorted), their TP flags, and for every GT the score
    of the prediction that matched it (-1 if unmatched).
    """
    keep = sorted((p for p in preds if p["score"] >= floor), key=lambda p: -p["score"])
    tp = np.zeros(len(keep), dtype=bool)
    gt_match_score = np.full(len(gts), -1.0)
    gt_by_class, pred_by_class = defaultdict(list), defaultdict(list)
    for j, g in enumerate(gts):
        gt_by_class[g["category_id"]].append(j)
    for i, p in enumerate(keep):
        pred_by_class[p["category_id"]].append(i)
    for cid, p_idx in pred_by_class.items():
        g_idx = gt_by_class.get(cid, [])
        if not g_idx:
            continue
        ious = iou_matrix([keep[i]["xyxy"] for i in p_idx], [gts[j]["xyxy"] for j in g_idx])
        used = np.zeros(len(g_idx), dtype=bool)
        for row, i in enumerate(p_idx):
            candidate = np.where(used, -1.0, ious[row])
            k = int(np.argmax(candidate))
            if candidate[k] >= CFG.evaluation_iou:
                used[k] = True
                tp[i] = True
                gt_match_score[g_idx[k]] = keep[i]["score"]
    return keep, tp, gt_match_score

def counts_at(keep, tp, gt_match_score, gts, threshold):
    """TP/FP/FN per category group at one confidence threshold (IoU 0.5)."""
    out = {}
    pred_cat = np.array([p["category_id"] for p in keep], dtype=int)
    pred_score = np.array([p["score"] for p in keep], dtype=float)
    gt_cat = np.array([g["category_id"] for g in gts], dtype=int)
    for group, ids in GROUPS.items():
        ids_arr = np.array(sorted(ids))
        pm = (pred_score >= threshold) & np.isin(pred_cat, ids_arr) if len(keep) else np.zeros(0, bool)
        gm = np.isin(gt_cat, ids_arr) if len(gts) else np.zeros(0, bool)
        out[group] = (int((tp & pm).sum()), int((~tp & pm).sum()),
                      int((gm & (gt_match_score < threshold)).sum()))
    return out

def prf(tp, fp, fn):
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return P, R, F

def image_signals(preds, flip_iou=None):
    """Image-level uncertainty signals computed ONLY from one inference pass.

    n_band       : detections just below the operating threshold (band_low <= s < operating)
    band_ratio   : n_band / (confident detections + 1)
    mean_u       : mean (1 - score) over detections >= analysis_floor
    mean_entropy : mean binary entropy of scores >= analysis_floor
    mean_u_flip  : mean (1 - flip IoU) over confident detections (only when flip data exist)
    small_pred_frac (NOT an uncertainty signal): fraction of predicted boxes < 32^2 px,
                   used as a size-prior baseline.
    """
    # Same ordering as match_image(), so flip_iou (aligned with its `keep`) lines up.
    kept = sorted((p for p in preds if p["score"] >= CFG.analysis_floor), key=lambda p: -p["score"])
    s = np.array([p["score"] for p in kept], dtype=float)
    areas = np.array([p["area"] for p in kept], dtype=float)
    n_conf = int((s >= CFG.operating_threshold).sum())
    n_band = int(((s >= CFG.band_low) & (s < CFG.operating_threshold)).sum())
    eps = 1e-6
    sc = np.clip(s, eps, 1 - eps)
    out = {
        "n_detections": int(s.size),
        "n_confident": n_conf,
        "n_band": n_band,
        "band_ratio": n_band / (n_conf + 1),
        "mean_u": float((1 - s).mean()) if s.size else 1.0,
        "mean_entropy": float((-(sc * np.log(sc) + (1 - sc) * np.log(1 - sc))).mean()) if s.size else float(np.log(2)),
        "small_pred_frac": float((areas < 32**2).mean()) if areas.size else 0.0,
    }
    if flip_iou is not None:
        conf_mask = s >= CFG.operating_threshold
        vals = np.asarray(flip_iou, float)[conf_mask] if conf_mask.size else np.zeros(0)
        out["mean_u_flip"] = float((1 - vals).mean()) if vals.size else 1.0
    return out

def make_split(rels):
    """Deterministic calibration / policy-test split of the evaluation images.

    With 548 val images and calibration_fraction=0.40 this gives 219 / 329.
    If CFG.split_csv points to an earlier split (columns image_relpath, split),
    that file is reused instead so all experiments share one split.
    """
    rels = sorted(rels)
    if CFG.split_csv:
        frame = pd.read_csv(CFG.split_csv)
        mapping = dict(zip(frame["image_relpath"], frame["split"]))
        if set(mapping) != set(rels):
            raise RuntimeError("split_csv does not cover exactly the analysed images.")
        if not set(mapping.values()) <= {"calibration", "policy_test"}:
            raise RuntimeError("split_csv split values must be calibration / policy_test.")
        return mapping
    order = np.random.default_rng(CFG.split_seed).permutation(len(rels))
    n_cal = int(round(len(rels) * CFG.calibration_fraction))
    calibration = {rels[i] for i in order[:n_cal]}
    return {r: ("calibration" if r in calibration else "policy_test") for r in rels}

def auroc(score, label):
    """Probability that a random positive (label 1) has a higher score than a random negative."""
    s = np.asarray(score, float)
    y = np.asarray(label, int)
    ok = np.isfinite(s)
    s, y = s[ok], y[ok]
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = pd.Series(s).rank(method="average").to_numpy()
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

In [7]:
def parse_spec(text):
    path, condition = text.split("::")
    family, run_id = path.split("/")
    return family, run_id, condition

KEYS = ["light"] + [f"strong{k + 1}" for k in range(len(CFG.strong))]
SPECS = dict(zip(KEYS, [CFG.light, *CFG.strong]))
COND = {key: load_condition(*parse_spec(spec)) for key, spec in SPECS.items()}
REF = COND["light"]["source"]

# All conditions must be evaluated on the same images with the same ground truth.
for key, cond in COND.items():
    src = cond["source"]
    if set(src["eval_rels"]) != set(REF["eval_rels"]):
        raise RuntimeError(f"{key}: evaluated on a different image list than the light condition.")
    for rel in REF["eval_rels"]:
        if len(src["gt"].get(rel, [])) != len(REF["gt"].get(rel, [])):
            raise RuntimeError(f"{key}: ground truth differs from the light condition for {rel}.")
    if COND["light"]["family"] != cond["family"]:
        print(f"WARNING: {key} uses model family {cond['family']} but light uses {COND['light']['family']}; "
              "adaptive computation is meant to escalate the SAME model.")

excluded = set()
if CFG.exclude_early_stopping_images:
    for cond in COND.values():
        excluded |= cond["source"]["early_stopping_rels"]
ANALYSIS_RELS = sorted(r for r in REF["eval_rels"] if r not in excluded)
SPLIT = make_split(ANALYSIS_RELS)
IDX_CAL = np.array([i for i, r in enumerate(ANALYSIS_RELS) if SPLIT[r] == "calibration"])
IDX_TEST = np.array([i for i, r in enumerate(ANALYSIS_RELS) if SPLIT[r] == "policy_test"])

GROUP_NAMES = ["all", "base", "novel"]
C = np.zeros((len(KEYS), len(ANALYSIS_RELS), 3, 3), dtype=np.int64)   # condition, image, group, (TP, FP, FN)
MS = np.zeros((len(KEYS), len(ANALYSIS_RELS)))
TILES = np.zeros((len(KEYS), len(ANALYSIS_RELS)))
for k, key in enumerate(KEYS):
    for i, rel in enumerate(tqdm(ANALYSIS_RELS, desc=f"Match {key}")):
        keep, tp, gms = match_image(COND[key]["preds"].get(rel, []), REF["gt"].get(rel, []), CFG.analysis_floor)
        counts = counts_at(keep, tp, gms, REF["gt"].get(rel, []), CFG.operating_threshold)
        C[k, i] = [counts[g] for g in GROUP_NAMES]
        MS[k, i] = float(COND[key]["latency"][rel]["pipeline_ms"])
        TILES[k, i] = float(COND[key]["latency"][rel].get("tiles", 1))

SIG_DF = pd.DataFrame([image_signals(COND["light"]["preds"].get(rel, [])) for rel in ANALYSIS_RELS])
SIG_DF.insert(0, "image_relpath", ANALYSIS_RELS)
SIG_DF.insert(1, "split", [SPLIT[r] for r in ANALYSIS_RELS])
# Net correct detections gained by escalating to strong[0]: (TP - FP)_strong - (TP - FP)_light.
GAIN = (C[1, :, 0, 0] - C[1, :, 0, 1]) - (C[0, :, 0, 0] - C[0, :, 0, 1])
SIG_DF["gain_strong1"] = GAIN

SOURCE_PROVENANCE = [{"key": key, "spec": SPECS[key], "folder": str(COND[key]["source"]["folder"]),
                      "source_clearml_task_id": COND[key]["source"]["config"].get("clearml_task_id"),
                      "source_run_mode": COND[key]["source"]["config"].get("config", {}).get("run_mode"),
                      "mean_ms": float(MS[k].mean()), "mean_tiles": float(TILES[k].mean())}
                     for k, key in enumerate(KEYS)]
print("Analysed images:", len(ANALYSIS_RELS), "| calibration:", len(IDX_CAL), "| policy-test:", len(IDX_TEST))
display(pd.DataFrame(SOURCE_PROVENANCE)[["key", "spec", "mean_ms", "mean_tiles"]])

Match light:   0%|          | 0/548 [00:00<?, ?it/s]

Match strong1:   0%|          | 0/548 [00:00<?, ?it/s]

Analysed images: 548 | calibration: 219 | policy-test: 329


,key,spec,mean_ms,mean_tiles
0,light,yoloe_26/20261028T153524Z_5dd65a8e::zero_shot__whole,80.434988,1.000000
1,strong1,yoloe_26/20261028T153524Z_5dd65a8e::zero_shot__sliced_overlap_20,742.549559,5.186131


## 4 · Policy machinery

A policy assigns every image a *level*: 0 = light only, k = also run strong k. Detection counts are additive over images, so dataset-level Precision / Recall / F1 (score ≥ 0.25, IoU ≥ 0.5) are recomputed exactly for any assignment. COCO AP is recomputed with pycocotools on the combined detections. Ties in integer signals (e.g. `n_band`) are broken by `mean_u` so that the rule is deterministic and deployable.

In [8]:
def summarize(idx, level, pays_light=True):
    """Metrics for images idx with per-image levels. pays_light=False means the strong pass replaces the light one."""
    idx = np.asarray(idx)
    level = np.asarray(level, dtype=int)
    total = C[level, idx].sum(axis=0)                      # (group, TP/FP/FN)
    out = {}
    for g, name in enumerate(GROUP_NAMES):
        P, R, F = prf(*total[g])
        out.update({f"{name}_TP": int(total[g, 0]), f"{name}_FP": int(total[g, 1]), f"{name}_FN": int(total[g, 2]),
                    f"{name}_Precision": P, f"{name}_Recall": R, f"{name}_F1": F})
    if pays_light:
        cost = MS[0, idx] + np.where(level > 0, MS[level, idx], 0.0)
        tiles = TILES[0, idx] + np.where(level > 0, TILES[level, idx], 0.0)
    else:
        cost, tiles = MS[level, idx], TILES[level, idx]
    out.update({"images": len(idx), "strong_fraction": float((level > 0).mean()) if len(idx) else 0.0,
                "mean_ms": float(cost.mean()), "fps": 1000 / float(cost.mean()), "mean_tiles": float(tiles.mean())})
    for k in range(1, len(KEYS)):
        out[f"fraction_{KEYS[k]}"] = float((level == k).mean()) if len(idx) else 0.0
    return out

def ordered(primary, secondary):
    """Indices sorted by primary descending, ties broken by secondary descending."""
    return np.lexsort((-np.asarray(secondary, float), -np.asarray(primary, float)))

def top_fraction_levels(n, order, fraction, strong_level=1):
    k = int(round(fraction * n))
    level = np.zeros(n, dtype=int)
    level[order[:k]] = strong_level
    return level

def composite(signal_name):
    """Deterministic tie-breaking: add a negligible multiple of mean_u (always in [0, 1])."""
    return SIG_DF[signal_name].to_numpy(float) + 1e-6 * SIG_DF["mean_u"].to_numpy(float)

coco_gt = COCO()
coco_gt.dataset = {k: v for k, v in REF["coco_gt"].items()}
with redirect_stdout(StringIO()):
    coco_gt.createIndex()
COCO_PRED = {key: {rel: [{"image_id": REF["rel_to_id"][rel], "category_id": p["category_id"],
                          "bbox": p["bbox"], "score": p["score"]} for p in COND[key]["preds"].get(rel, [])]
                   for rel in ANALYSIS_RELS} for key in KEYS}

def valid_mean(values):
    values = np.asarray(values)
    valid = values[values > -1]
    return float(valid.mean()) if valid.size else np.nan

def coco_ap(idx, level):
    image_ids = [REF["rel_to_id"][ANALYSIS_RELS[i]] for i in idx]
    detections = [d for i, l in zip(idx, level) for d in COCO_PRED[KEYS[l]][ANALYSIS_RELS[i]]]
    wanted = set(image_ids)
    gt_counts = Counter(a["category_id"] for a in coco_gt.dataset["annotations"] if a["image_id"] in wanted)
    if not detections:
        return {f"{g}_AP": 0.0 for g in GROUP_NAMES} | {f"{g}_AP50": 0.0 for g in GROUP_NAMES}
    with redirect_stdout(StringIO()):
        coco_dt = coco_gt.loadRes(detections)
        ev = COCOeval(coco_gt, coco_dt, "bbox")
        ev.params.catIds = CATEGORY_IDS
        ev.params.imgIds = image_ids
        ev.params.maxDets = [1, 10, CFG.max_detections_per_image]
        ev.evaluate()
        ev.accumulate()
    precision = ev.eval["precision"]
    out = {}
    for name, ids in (("all", CATEGORY_IDS), ("base", sorted(BASE_IDS)), ("novel", sorted(NOVEL_IDS))):
        cols = [CATEGORY_IDS.index(c) for c in ids if gt_counts[c] > 0]
        out[f"{name}_AP"] = valid_mean(precision[:, :, cols, 0, -1]) if cols else np.nan
        out[f"{name}_AP50"] = valid_mean(precision[0, :, cols, 0, -1]) if cols else np.nan
    return out

FRACTIONS = np.round(np.arange(0, 1 + 1e-9, CFG.budget_step), 6)
rng = np.random.default_rng(CFG.seed)

## 5 · Choose the signal and threshold on calibration images only

For each candidate signal we escalate the top 0%, 5%, …, 100% most uncertain calibration images and record F1. The signal with the largest mean F1 over this curve is chosen (unless `signal` names one). The threshold is the smallest fraction that keeps `target_gain_retention` of the F1 gain of always running strong[0]; it is then frozen as a **signal value** and applied unchanged to the policy-test images.

In [9]:
n_cal = len(IDX_CAL)
selection_rows, cal_curves = [], {}
for name in CFG.candidate_signals:
    order = ordered(composite(name)[IDX_CAL], SIG_DF["mean_u"].to_numpy()[IDX_CAL])
    curve = [summarize(IDX_CAL, top_fraction_levels(n_cal, order, f))["all_F1"] for f in FRACTIONS]
    cal_curves[name] = curve
    selection_rows.append({"signal": name, "mean_F1_over_budget_curve": float(np.mean(curve)),
                           "spearman_with_gain": float(pd.Series(SIG_DF[name].to_numpy()[IDX_CAL]).corr(
                               pd.Series(GAIN[IDX_CAL]), method="spearman"))})
SIGNAL_SELECTION_DF = pd.DataFrame(selection_rows).sort_values("mean_F1_over_budget_curve", ascending=False)
SIGNAL = SIGNAL_SELECTION_DF["signal"].iloc[0] if CFG.signal == "auto" else CFG.signal
if SIGNAL not in SIG_DF:
    raise ValueError(f"Unknown signal {SIGNAL!r}")
SCORE = composite(SIGNAL)

order_cal = ordered(SCORE[IDX_CAL], SIG_DF["mean_u"].to_numpy()[IDX_CAL])
f1_cal = np.array([summarize(IDX_CAL, top_fraction_levels(n_cal, order_cal, f))["all_F1"] for f in FRACTIONS])
full_gain = f1_cal[-1] - f1_cal[0]
if full_gain > 0:
    chosen_fraction = float(FRACTIONS[np.argmax(f1_cal - f1_cal[0] >= CFG.target_gain_retention * full_gain - 1e-12)])
    rule = f"smallest fraction keeping >= {CFG.target_gain_retention:.0%} of the full-slicing F1 gain"
else:
    chosen_fraction = float(FRACTIONS[int(np.argmax(f1_cal))])
    rule = "always slicing does not improve calibration F1; fraction with the best calibration F1"
k_cal = int(round(chosen_fraction * n_cal))
THRESHOLD = float(np.inf) if k_cal == 0 else float(SCORE[IDX_CAL][order_cal[k_cal - 1]])
print(f"Signal: {SIGNAL} | calibration fraction escalated: {chosen_fraction:.2f} | threshold: {THRESHOLD:.6g}")
print("Rule:", rule)
display(SIGNAL_SELECTION_DF)

Signal: n_band | calibration fraction escalated: 0.60 | threshold: 11
Rule: smallest fraction keeping >= 90% of the full-slicing F1 gain


,signal,mean_F1_over_budget_curve,spearman_with_gain
0,n_band,0.337412,-0.207115
1,band_ratio,0.333876,0.040298
2,mean_u,0.331588,0.144227
3,mean_entropy,0.327128,-0.151931


## 6 · Evaluate all policies on the policy-test images

`random_same_fraction` reports the mean over 1000 random draws with a 95% interval, and `p_random_at_least_as_good` = share of random draws whose F1 is at least the adaptive F1 (small = the signal beats chance). AP for the random policy is averaged over 20 draws.

In [10]:
n_test = len(IDX_TEST)
score_test = SCORE[IDX_TEST]
policies = {}   # name -> (level array over IDX_TEST, pays_light)

policies["light_only"] = (np.zeros(n_test, dtype=int), True)
for k in range(1, len(KEYS)):
    policies[f"strong_only_{KEYS[k]}"] = (np.full(n_test, k), False)
adaptive_level = np.where(score_test >= THRESHOLD, 1, 0)
policies["adaptive_uncertainty"] = (adaptive_level, True)
k_test = int(adaptive_level.sum())

size_order = ordered(SIG_DF["small_pred_frac"].to_numpy()[IDX_TEST], SIG_DF["mean_u"].to_numpy()[IDX_TEST])
level = np.zeros(n_test, dtype=int)
level[size_order[:k_test]] = 1
policies["size_prior_same_fraction"] = (level, True)
oracle_order = np.argsort(-GAIN[IDX_TEST], kind="stable")
level = np.zeros(n_test, dtype=int)
level[oracle_order[:k_test]] = 1
policies["oracle_same_fraction"] = (level, True)
policies["oracle_unbounded"] = ((GAIN[IDX_TEST] > 0).astype(int), True)

# Graded overlap: cheapest strong for medium uncertainty, most expensive for high uncertainty.
GRADED_INFO = {}
if len(KEYS) >= 3:
    strong_levels = list(range(1, len(KEYS)))
    cheap = min(strong_levels, key=lambda k: MS[k, IDX_CAL].mean())
    dear = max(strong_levels, key=lambda k: MS[k, IDX_CAL].mean())
    if cheap != dear:
        binary_cal_cost = summarize(IDX_CAL, np.where(SCORE[IDX_CAL] >= THRESHOLD, 1, 0))["mean_ms"]
        best = None
        for f_hi in FRACTIONS:
            for f_mid in FRACTIONS:
                if f_hi + f_mid > 1 + 1e-9:
                    continue
                k_hi, k_mid = int(round(f_hi * n_cal)), int(round(f_mid * n_cal))
                lv = np.zeros(n_cal, dtype=int)
                lv[order_cal[:k_hi]] = dear
                lv[order_cal[k_hi:k_hi + k_mid]] = cheap
                m = summarize(IDX_CAL, lv)
                if m["mean_ms"] <= binary_cal_cost + 1e-9 and (best is None or m["all_F1"] > best[0] + 1e-12):
                    best = (m["all_F1"], k_hi, k_mid)
        _, k_hi, k_mid = best
        ranked = SCORE[IDX_CAL][order_cal]
        tau_hi = np.inf if k_hi == 0 else float(ranked[k_hi - 1])
        tau_mid = tau_hi if k_mid == 0 else float(ranked[k_hi + k_mid - 1])
        level = np.where(score_test >= tau_hi, dear, np.where(score_test >= tau_mid, cheap, 0))
        policies["graded_overlap"] = (level, True)
        GRADED_INFO = {"cheap_level": KEYS[cheap], "expensive_level": KEYS[dear], "tau_high": tau_hi,
                       "tau_mid": tau_mid, "calibration_cost_cap_ms": binary_cal_cost,
                       "calibration_fraction_high": k_hi / n_cal, "calibration_fraction_mid": k_mid / n_cal}
        print("Graded overlap:", GRADED_INFO)

policy_rows = []
for name, (lv, pays) in policies.items():
    row = {"policy": name, **summarize(IDX_TEST, lv, pays), **coco_ap(IDX_TEST, lv)}
    policy_rows.append(row)

random_f1, random_recall, random_cost, random_rows = [], [], [], []
for d in range(CFG.random_draws):
    lv = np.zeros(n_test, dtype=int)
    lv[rng.choice(n_test, size=k_test, replace=False)] = 1
    m = summarize(IDX_TEST, lv)
    random_f1.append(m["all_F1"]); random_recall.append(m["all_Recall"]); random_cost.append(m["mean_ms"])
    random_rows.append(m)
ap_draws = []
for d in range(CFG.random_ap_draws):
    lv = np.zeros(n_test, dtype=int)
    lv[rng.choice(n_test, size=k_test, replace=False)] = 1
    ap_draws.append(coco_ap(IDX_TEST, lv))
random_mean = pd.DataFrame(random_rows).mean(numeric_only=True).to_dict()
random_mean.update(pd.DataFrame(ap_draws).mean(numeric_only=True).to_dict())
adaptive_f1 = next(r for r in policy_rows if r["policy"] == "adaptive_uncertainty")["all_F1"]
random_mean.update({"policy": "random_same_fraction",
                    "all_F1_CI_low": float(np.percentile(random_f1, 2.5)),
                    "all_F1_CI_high": float(np.percentile(random_f1, 97.5)),
                    "p_random_at_least_as_good": float(np.mean(np.asarray(random_f1) >= adaptive_f1 - 1e-12))})
policy_rows.append(random_mean)

POLICY_DF = pd.DataFrame(policy_rows)
strong1_ms = POLICY_DF.loc[POLICY_DF["policy"] == "strong_only_strong1", "mean_ms"].iloc[0]
light_f1 = POLICY_DF.loc[POLICY_DF["policy"] == "light_only", "all_F1"].iloc[0]
strong1_f1 = POLICY_DF.loc[POLICY_DF["policy"] == "strong_only_strong1", "all_F1"].iloc[0]
POLICY_DF["cost_relative_to_strong1"] = POLICY_DF["mean_ms"] / strong1_ms
POLICY_DF["F1_gain_retained_vs_strong1"] = ((POLICY_DF["all_F1"] - light_f1) / (strong1_f1 - light_f1)
                                            if strong1_f1 != light_f1 else np.nan)
display(POLICY_DF[["policy", "strong_fraction", "all_F1", "all_Recall", "novel_Recall", "all_AP", "novel_AP",
                   "mean_ms", "cost_relative_to_strong1", "F1_gain_retained_vs_strong1"]])

,policy,strong_fraction,all_F1,all_Recall,novel_Recall,all_AP,novel_AP,mean_ms,cost_relative_to_strong1,F1_gain_retained_vs_strong1
0,light_only,0.000000,0.337227,0.221305,0.071705,0.108644,0.088840,79.314697,0.107277,0.000000
1,strong_only_strong1,1.000000,0.358130,0.260751,0.107558,0.108353,0.071730,739.345884,1.000000,1.000000
2,adaptive_uncertainty,0.544073,0.358855,0.254777,0.095930,0.109275,0.078751,586.861982,0.793758,1.034716
3,size_prior_same_fraction,0.544073,0.355982,0.248320,0.094961,0.110832,0.082789,461.811428,0.624622,0.897270
4,oracle_same_fraction,0.544073,0.362710,0.248671,0.092054,0.115523,0.092923,425.744157,0.575839,1.219133
5,oracle_unbounded,0.164134,0.354176,0.236503,0.083333,0.115454,0.098621,188.710977,0.255240,0.810833
6,random_same_fraction,0.544073,0.349200,0.242818,0.091328,0.106986,0.079263,481.518438,0.651276,0.572821


## 7 · Paired bootstrap of the differences (policy-test images)

Images are resampled with replacement and both policies are re-scored on the same resample. ΔF1 > 0 with a CI that excludes 0 is a reliable improvement; `P(Δ≤0)` is the share of resamples where the first policy is not better.

In [11]:
def paired_bootstrap(first, second):
    (lv_a, pay_a), (lv_b, pay_b) = policies[first], policies[second]
    values = defaultdict(list)
    for _ in range(CFG.bootstrap_samples):
        pick = rng.integers(0, n_test, n_test)
        a = summarize(IDX_TEST[pick], lv_a[pick], pay_a)
        b = summarize(IDX_TEST[pick], lv_b[pick], pay_b)
        for metric in ("all_F1", "all_Recall", "novel_Recall", "mean_ms"):
            values[metric].append(a[metric] - b[metric])
    point_a, point_b = summarize(IDX_TEST, lv_a, pay_a), summarize(IDX_TEST, lv_b, pay_b)
    row = {"comparison": f"{first} - {second}"}
    for metric, vals in values.items():
        vals = np.asarray(vals)
        row[f"delta_{metric}"] = point_a[metric] - point_b[metric]
        row[f"delta_{metric}_CI_low"], row[f"delta_{metric}_CI_high"] = np.percentile(vals, [2.5, 97.5])
        row[f"P(delta_{metric}<=0)"] = float(np.mean(vals <= 0))
    return row

pairs = [("adaptive_uncertainty", "light_only"), ("adaptive_uncertainty", "strong_only_strong1"),
         ("adaptive_uncertainty", "size_prior_same_fraction"), ("strong_only_strong1", "light_only")]
if "graded_overlap" in policies:
    pairs.append(("graded_overlap", "adaptive_uncertainty"))
BOOT_DF = pd.DataFrame([paired_bootstrap(a, b) for a, b in tqdm(pairs, desc="Paired bootstrap")])
display(BOOT_DF[["comparison", "delta_all_F1", "delta_all_F1_CI_low", "delta_all_F1_CI_high",
                 "delta_mean_ms", "P(delta_all_F1<=0)"]])

Paired bootstrap:   0%|          | 0/4 [00:00<?, ?it/s]

,comparison,delta_all_F1,delta_all_F1_CI_low,delta_all_F1_CI_high,delta_mean_ms,P(delta_all_F1<=0)
0,adaptive_uncertainty - light_only,0.021629,0.015786,0.028040,507.547285,0.000
1,adaptive_uncertainty - strong_only_strong1,0.000726,-0.002125,0.004093,-152.483902,0.309
2,adaptive_uncertainty - size_prior_same_fraction,0.002873,-0.002061,0.007696,125.050554,0.118
3,strong_only_strong1 - light_only,0.020903,0.013820,0.026955,660.031187,0.000


## 8 · Budget curves: accuracy at every escalation fraction

Escalate the top x% of policy-test images by each ranking (uncertainty, size prior, oracle) or at random, for x = 0…100%. A good signal rises quickly towards the always-sliced F1 and stays well above the random line.

In [12]:
rankings = {
    f"uncertainty ({SIGNAL})": ordered(score_test, SIG_DF["mean_u"].to_numpy()[IDX_TEST]),
    "size prior": size_order,
    "oracle": oracle_order,
}
curve_rows = []
for f in FRACTIONS:
    for name, order in rankings.items():
        m = summarize(IDX_TEST, top_fraction_levels(n_test, order, f))
        curve_rows.append({"ranking": name, "fraction": f, **{k: m[k] for k in
                           ("all_F1", "all_Recall", "novel_F1", "novel_Recall", "mean_ms")}})
    draws = []
    for _ in range(CFG.random_curve_draws):
        lv = np.zeros(n_test, dtype=int)
        lv[rng.choice(n_test, size=int(round(f * n_test)), replace=False)] = 1
        draws.append(summarize(IDX_TEST, lv))
    frame = pd.DataFrame(draws)
    curve_rows.append({"ranking": "random", "fraction": f, "all_F1": frame["all_F1"].mean(),
                       "all_Recall": frame["all_Recall"].mean(), "novel_F1": frame["novel_F1"].mean(),
                       "novel_Recall": frame["novel_Recall"].mean(), "mean_ms": frame["mean_ms"].mean(),
                       "all_F1_CI_low": frame["all_F1"].quantile(0.025),
                       "all_F1_CI_high": frame["all_F1"].quantile(0.975)})
CURVE_DF = pd.DataFrame(curve_rows)
CAL_CURVE_DF = pd.DataFrame([{"signal": s, "fraction": f, "all_F1": v}
                             for s, curve in cal_curves.items() for f, v in zip(FRACTIONS, curve)])

FIGURE_FILES = []
STYLE = {"random": ("#999999", "--"), "size prior": ("#f2a65a", "-"), "oracle": ("#6a994e", ":")}
for metric, label in (("all_F1", "F1 (all classes)"), ("novel_Recall", "Recall (novel classes)")):
    fig, ax = plt.subplots(figsize=(6, 4))
    for name, sub in CURVE_DF.groupby("ranking", sort=False):
        color, style = STYLE.get(name, ("#2a6f97", "-"))
        ax.plot(sub["fraction"], sub[metric], linestyle=style, color=color, label=name, linewidth=2 if color == "#2a6f97" else 1.5)
        if name == "random" and metric == "all_F1":
            ax.fill_between(sub["fraction"], sub["all_F1_CI_low"], sub["all_F1_CI_high"], color=color, alpha=0.15)
    chosen = POLICY_DF.loc[POLICY_DF["policy"] == "adaptive_uncertainty"].iloc[0]
    ax.scatter([chosen["strong_fraction"]], [chosen[metric]], color="#bc4749", zorder=5, label="chosen threshold")
    ax.set(xlabel="Fraction of images escalated to strong[0]", ylabel=label, title=f"Budget curve · {label}")
    ax.legend(frameon=False, fontsize=8)
    FIGURE_FILES.append(log_figure("Budget curve", metric, fig, f"budget_curve_{metric}.png"))

fig, ax = plt.subplots(figsize=(6, 4))
sub = CURVE_DF[CURVE_DF["ranking"] == f"uncertainty ({SIGNAL})"]
ax.plot(sub["mean_ms"], sub["all_F1"], color="#2a6f97", label=f"uncertainty ({SIGNAL}) sweep")
sub = CURVE_DF[CURVE_DF["ranking"] == "random"]
ax.plot(sub["mean_ms"], sub["all_F1"], color="#999999", linestyle="--", label="random sweep")
for _, row in POLICY_DF.iterrows():
    if row["policy"] in ("light_only", "adaptive_uncertainty", "graded_overlap") or row["policy"].startswith("strong_only"):
        ax.scatter(row["mean_ms"], row["all_F1"], zorder=5)
        offset = (4, -10) if row["policy"] == "graded_overlap" else (4, 4)
        ax.annotate(row["policy"], (row["mean_ms"], row["all_F1"]), fontsize=7, xytext=offset, textcoords="offset points")
ax.set(xlabel="Mean pipeline latency per image (ms)", ylabel="F1 (all classes)", title="Accuracy vs computation")
ax.legend(frameon=False, fontsize=8)
FIGURE_FILES.append(log_figure("Accuracy vs computation", "policy-test", fig, "accuracy_vs_cost.png"))
print("Figures:", len(FIGURE_FILES))

Figures: 3


## 9 · Keep **all** results locally; log compact tables and the file inventory to ClearML

`results/exp3_adaptive_computation/<run_id>/` keeps the policy table, bootstrap table, budget curves, the per-image decisions with signals and gains, the split, figures and configuration.

In [13]:
decision_df = SIG_DF.copy()
decision_df["latency_light_ms"] = MS[0]
for k in range(1, len(KEYS)):
    decision_df[f"latency_{KEYS[k]}_ms"] = MS[k]
decision_df["selected_signal_value"] = SCORE
for name, (lv, _) in policies.items():
    column = np.full(len(ANALYSIS_RELS), np.nan)
    column[IDX_TEST] = lv
    decision_df[f"level_{name}"] = column

write_csv("policies_policy_test.csv", POLICY_DF)
write_csv("paired_bootstrap.csv", BOOT_DF)
write_csv("budget_curves_policy_test.csv", CURVE_DF)
write_csv("budget_curves_calibration.csv", CAL_CURVE_DF)
write_csv("signal_selection_calibration.csv", SIGNAL_SELECTION_DF)
write_csv("per_image_decisions.csv", decision_df)
write_csv("image_split.csv", pd.DataFrame({"image_relpath": list(SPLIT), "split": list(SPLIT.values())}))
write_json("config_and_protocol.json", {"config": asdict(CFG), "run_id": RUN_ID, "clearml_task_id": task.id,
           "protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
           "category_ids": CATEGORY_IDS, "base_names": BASE_NAMES, "novel_names": NOVEL_NAMES,
           "sources": SOURCE_PROVENANCE, "selected_signal": SIGNAL, "threshold": THRESHOLD,
           "calibration_fraction_escalated": chosen_fraction, "threshold_rule": rule, "graded": GRADED_INFO,
           "analysis_image_relpaths": ANALYSIS_RELS, "split_counts": dict(Counter(SPLIT.values())),
           "platform": platform.platform()})

for name, frame in (("Policies (policy-test)", POLICY_DF), ("Paired bootstrap", BOOT_DF),
                    ("Signal selection (calibration)", SIGNAL_SELECTION_DF), ("Budget curves", CURVE_DF)):
    log_table(name, frame)
HEADLINE = {"signal": SIGNAL, "threshold": THRESHOLD}
for _, row in POLICY_DF.iterrows():
    for metric in ("all_F1", "all_Recall", "novel_Recall", "all_AP", "novel_AP", "mean_ms", "strong_fraction",
                   "cost_relative_to_strong1"):
        if pd.notna(row.get(metric)):
            logger.report_scalar(title=metric, series=row["policy"], value=float(row[metric]), iteration=0)
            HEADLINE[f"{row['policy']}|{metric}"] = row[metric]
HEADLINE["p_random_at_least_as_good"] = random_mean["p_random_at_least_as_good"]
NOTE_TEXT = ("Experiment 3 note: offline simulation from saved Experiment 1 predictions and per-image latency. "
             "Escalated images pay light + strong latency. Signal and threshold chosen on calibration images only; "
             "all reported numbers are on policy-test images. P/R/F1 at score 0.25, IoU 0.5; AP is COCO AP@[.50:.95]. "
             "GPU memory is not measured offline.")

In [14]:
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file() and path.name != "local_inventory.json":
        LOCAL_INDEX["results"][path.relative_to(RESULTS_DIR).as_posix()] = record_local_file(path)

inventory = {"run_id": RUN_ID, "experiment": CFG.task_name_prefix,
             "protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
             "clearml_task_id": task.id, "local_results_dir": str(RESULTS_DIR),
             "inventory_path": str(RESULTS_DIR / "local_inventory.json"),
             "sources": SOURCE_PROVENANCE, "objects": LOCAL_INDEX,
             "headline_metrics": HEADLINE}
manifest_file = RESULTS_DIR / "local_inventory.json"
manifest_file.write_text(json.dumps(clean_json(inventory), indent=2, allow_nan=False) + "\n", encoding="utf-8")
if manifest_file.stat().st_size >= 2 * 1024 * 1024:
    raise RuntimeError("ClearML inventory unexpectedly exceeds 2 MiB; inspect before uploading.")
task.upload_artifact(name="local_inventory_small", artifact_object=str(manifest_file))
logger.report_text(f"All complete results saved locally: {RESULTS_DIR}")
logger.report_text(f"Protocol Dataset ID: {PROTOCOL_ID} | SHA256 fingerprint: {FINGERPRINT}")
logger.report_text(NOTE_TEXT)
logger.flush(wait=True)
task.close()
print("DONE. ClearML project:", CFG.clearml_project)
print("ClearML task:", task.id)
print("Local results:", RESULTS_DIR)
print("Free space (GiB):", round(shutil.disk_usage(ROOT).free / 1024**3, 2))
print("IMPORTANT:", "SMOKE SOURCES — pipeline check only, not for the paper" if CFG.allow_smoke_sources
      else "Full-mode sources analysed")

All complete results saved locally: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/exp3_adaptive_computation/20261029T130521Z_615f90ed
Protocol Dataset ID: 738dce9e1a5440dab34195418a997a20 | SHA256 fingerprint: 158cb2c904016fa324c236b998393b1a94c176a4eabedb159c76bdeffc66c195
Experiment 3 note: offline simulation from saved Experiment 1 predictions and per-image latency. Escalated images pay light + strong latency. Signal and threshold chosen on calibration images only; all reported numbers are on policy-test images. P/R/F1 at score 0.25, IoU 0.5; AP is COCO AP@[.50:.95]. GPU memory is not measured offline.
DONE. ClearML project: SageMaker-Aerial-OVD-2026/Uncertainty-Adaptive-Computation-2026
ClearML task: dc3b63d99f024fb8a7516d277a0824f1
Local results: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/exp3_adaptive_computation/20261029T130521Z_615f90ed
Free space (GiB): 6.54
IMPORTANT: Full-mode sources analysed
